# 7E. Multimodal Pose + RGB Fusion Colab


## Reason, Approach, Result Interpretation

**Why this stage exists**

The pose and RGB branches are now informative but exercise-dependent. `squat` remains clearly pose-first, `push_up` is RGB-friendly, and `pull_up` is mixed. This stage tests whether a small multimodal fusion model can reuse the strengths of both representations without jumping to a much larger video-native architecture.

**Approach**

This stage trains a simple late-fusion TCN that reads both normalized pose sequences and frozen RGB feature sequences. To keep the experiment aligned with the evidence already collected, it uses the strongest single-modality RGB source per exercise:

- `squat`: `ResNet50` RGB features from `7B`
- `pull_up`: `ResNet50` RGB features from `7B`
- `push_up`: `ResNet18` RGB features from Stage `7`

The fusion branch is then compared directly against:

- the shared pose `6B` baseline
- the best RGB reference for that exercise
- and for `squat`, the frozen dedicated pose baseline `squat_tcn_l1_channels96`

**How to interpret the result**

If multimodal fusion beats both single-modality branches on an exercise, then the extra complexity is justified there. If it only matches the better branch or degrades both, then the modalities are not combining productively enough to earn the extra architecture.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

POSE_TRAIN_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
RGB_TRAIN_REL = Path('artifacts/3_Modeling/train_rgb_count_tcn.py')
MULTI_TRAIN_REL = Path('artifacts/3_Modeling/train_multimodal_count_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')

def sync_drive_file(rel: Path) -> None:
    src = CODE_ROOT / rel
    dst = DRIVE_PROJECT_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists() and not dst.exists():
        print(f'[sync] missing both copies: {rel}')
        return
    if not src.exists():
        print(f'[sync] keeping Drive copy (no /content source): {rel}')
        return
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f'[sync] copied /content -> Drive (Drive missing): {rel}')
        return
    if src.stat().st_mtime > dst.stat().st_mtime + 1.0:
        shutil.copy2(src, dst)
        print(f'[sync] copied newer /content -> Drive: {rel}')
    else:
        print(f'[sync] keeping Drive copy (newer or equal): {rel}')

for rel in [POSE_TRAIN_REL, RGB_TRAIN_REL, MULTI_TRAIN_REL, COMPARE_REL]:
    sync_drive_file(rel)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
POSE_SEQUENCE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'
RGB18_INDEX = ANNOTATION_DIR / 'rgb_feature_index_selected.csv'
RGB50_INDEX = ANNOTATION_DIR / 'rgb_feature_index_resnet50_selected.csv'

print('POSE_SEQUENCE_INDEX =', POSE_SEQUENCE_INDEX, POSE_SEQUENCE_INDEX.exists())
print('RGB18_INDEX =', RGB18_INDEX, RGB18_INDEX.exists())
print('RGB50_INDEX =', RGB50_INDEX, RGB50_INDEX.exists())
print('Multimodal trainer present =', (DRIVE_PROJECT_ROOT / MULTI_TRAIN_REL).exists())


## Multimodal Preset


In [ ]:
import pandas as pd

FUSION_RUNS = [
    {
        'exercise': 'squat',
        'seq_len': 256,
        'pose_run': 'pose_count_tcn_squat_seq256',
        'pose_best_run': 'squat_tcn_l1_channels96',
        'rgb_reference_variant': 'rgb_stronger',
        'rgb_reference_run': 'rgb_resnet50_count_tcn_squat_seq256',
        'rgb_backbone': 'resnet50',
        'rgb_index_csv': RGB50_INDEX,
        'fusion_run': 'multimodal_pose_rgb_squat_seq256',
    },
    {
        'exercise': 'pull_up',
        'seq_len': 192,
        'pose_run': 'pose_count_tcn_pull_up_seq192',
        'rgb_reference_variant': 'rgb_stronger',
        'rgb_reference_run': 'rgb_resnet50_count_tcn_pull_up_seq192',
        'rgb_backbone': 'resnet50',
        'rgb_index_csv': RGB50_INDEX,
        'fusion_run': 'multimodal_pose_rgb_pull_up_seq192',
    },
    {
        'exercise': 'push_up',
        'seq_len': 128,
        'pose_run': 'pose_count_tcn_push_up_seq128',
        'rgb_reference_variant': 'rgb_stage7',
        'rgb_reference_run': 'rgb_count_tcn_push_up_seq128',
        'rgb_backbone': 'resnet18',
        'rgb_index_csv': RGB18_INDEX,
        'fusion_run': 'multimodal_pose_rgb_push_up_seq128',
    },
]

subset_exercises = [cfg['exercise'] for cfg in FUSION_RUNS]
subset_counts = pd.read_csv(POSE_SEQUENCE_INDEX)
subset_counts = subset_counts[subset_counts['type'].isin(subset_exercises)]
subset_counts = subset_counts.groupby(['type', 'split']).size().unstack(fill_value=0).sort_index()
display(subset_counts)
display(pd.DataFrame(FUSION_RUNS)[['exercise', 'seq_len', 'rgb_backbone', 'rgb_reference_variant', 'rgb_reference_run', 'fusion_run']])


## Multimodal Training Execution


In [ ]:
import subprocess
import pandas as pd

training_failures = []
for cfg in FUSION_RUNS:
    cmd = [
        'python', '-u', str(DRIVE_PROJECT_ROOT / MULTI_TRAIN_REL),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--pose-index-csv', str(POSE_SEQUENCE_INDEX),
        '--rgb-index-csv', str(cfg['rgb_index_csv']),
        '--run-name', cfg['fusion_run'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', '80',
        '--batch-size', '12',
        '--lr', '0.001',
        '--weight-decay', '0.0001',
        '--pose-channels', '48',
        '--rgb-channels', '48',
        '--fusion-channels', '128',
        '--kernel-size', '3',
        '--num-blocks', '4',
        '--dropout', '0.20',
        '--patience', '15',
        '--loss', 'l1',
        '--eval-transform', 'raw',
        '--selection-metric', 'mae',
        '--sampler', 'balanced_count',
        '--time-warp-range', '0.12',
        '--pose-feature-noise-std', '0.02',
        '--rgb-feature-noise-std', '0.02',
        '--frame-dropout-prob', '0.03',
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'run_name': cfg['fusion_run'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['exercise']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All multimodal runs completed.')


## Raw Metric Review


In [ ]:
import json
import pandas as pd

rows = []
summary_rows = []
for cfg in FUSION_RUNS:
    variant_specs = [
        ('pose_6B', cfg['pose_run']),
        (cfg['rgb_reference_variant'], cfg['rgb_reference_run']),
        ('multimodal_fusion', cfg['fusion_run']),
    ]
    if cfg.get('pose_best_run'):
        variant_specs.insert(1, ('pose_best_squat', cfg['pose_best_run']))

    metrics_by_variant = {}
    for variant, run_name in variant_specs:
        metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        row = {
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'best_epoch': metrics.get('best_epoch'),
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_rmse': metrics['valid_metrics']['rmse'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        }
        rows.append(row)
        metrics_by_variant[variant] = row

    if 'multimodal_fusion' not in metrics_by_variant:
        continue

    fusion_row = metrics_by_variant['multimodal_fusion']
    summary_row = {
        'exercise': cfg['exercise'],
        'seq_len': cfg['seq_len'],
        'rgb_reference_variant': cfg['rgb_reference_variant'],
        'valid_mae_fusion': fusion_row['valid_mae'],
        'valid_within_1_fusion': fusion_row['valid_within_1'],
    }
    if 'pose_6B' in metrics_by_variant:
        summary_row['valid_mae_pose_6B'] = metrics_by_variant['pose_6B']['valid_mae']
        summary_row['valid_within_1_pose_6B'] = metrics_by_variant['pose_6B']['valid_within_1']
        summary_row['delta_mae_fusion_minus_pose_6B'] = fusion_row['valid_mae'] - metrics_by_variant['pose_6B']['valid_mae']
        summary_row['delta_within_1_fusion_minus_pose_6B'] = fusion_row['valid_within_1'] - metrics_by_variant['pose_6B']['valid_within_1']
    if cfg['rgb_reference_variant'] in metrics_by_variant:
        rgb_ref_row = metrics_by_variant[cfg['rgb_reference_variant']]
        summary_row['valid_mae_rgb_reference'] = rgb_ref_row['valid_mae']
        summary_row['valid_within_1_rgb_reference'] = rgb_ref_row['valid_within_1']
        summary_row['delta_mae_fusion_minus_rgb_reference'] = fusion_row['valid_mae'] - rgb_ref_row['valid_mae']
        summary_row['delta_within_1_fusion_minus_rgb_reference'] = fusion_row['valid_within_1'] - rgb_ref_row['valid_within_1']
    if 'pose_best_squat' in metrics_by_variant:
        best_pose_row = metrics_by_variant['pose_best_squat']
        summary_row['valid_mae_pose_best_squat'] = best_pose_row['valid_mae']
        summary_row['valid_within_1_pose_best_squat'] = best_pose_row['valid_within_1']
        summary_row['delta_mae_fusion_minus_best_squat'] = fusion_row['valid_mae'] - best_pose_row['valid_mae']
        summary_row['delta_within_1_fusion_minus_best_squat'] = fusion_row['valid_within_1'] - best_pose_row['valid_within_1']
    summary_rows.append(summary_row)

compare_df = pd.DataFrame(rows)
display(compare_df.sort_values(['exercise', 'variant']))
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    display(summary_df.sort_values('exercise'))


## Baseline-Comparison Review


In [ ]:
import subprocess
import json
import pandas as pd

comparison_failures = []
for cfg in FUSION_RUNS:
    run_dir = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['fusion_run']
    pred_path = run_dir / 'predictions.csv'
    if not pred_path.exists():
        continue
    summary_path = run_dir / 'baseline_comparison_summary.json'
    if not summary_path.exists():
        cmd = [
            'python', str(DRIVE_PROJECT_ROOT / COMPARE_REL),
            '--index-csv', str(POSE_SEQUENCE_INDEX),
            '--predictions-csv', str(pred_path),
            '--exercise', cfg['exercise'],
        ]
        print('\nComparing:', ' '.join(cmd))
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as exc:
            comparison_failures.append({
                'exercise': cfg['exercise'],
                'run_name': cfg['fusion_run'],
                'returncode': exc.returncode,
            })

rows = []
summary_rows = []
for cfg in FUSION_RUNS:
    variant_specs = [
        ('pose_6B', cfg['pose_run']),
        (cfg['rgb_reference_variant'], cfg['rgb_reference_run']),
        ('multimodal_fusion', cfg['fusion_run']),
    ]
    if cfg.get('pose_best_run'):
        variant_specs.insert(1, ('pose_best_squat', cfg['pose_best_run']))

    metrics_by_variant = {}
    for variant, run_name in variant_specs:
        summary_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'baseline_comparison_summary.json'
        if not summary_path.exists():
            continue
        with open(summary_path, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        row = {
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'model_mae': summary['model_metrics']['mae'],
            'baseline_mae': summary['baseline_metrics']['mae'],
            'delta_mae_vs_trivial': summary['delta_vs_baseline']['mae'],
            'model_within_1': summary['model_metrics']['within_1'],
            'baseline_within_1': summary['baseline_metrics']['within_1'],
            'delta_within_1_vs_trivial': summary['delta_vs_baseline']['within_1'],
            'model_beats_baseline_rows': summary['row_level']['model_beats_baseline'],
            'valid_rows': summary['row_level']['valid_rows'],
        }
        rows.append(row)
        metrics_by_variant[variant] = row

    if 'multimodal_fusion' not in metrics_by_variant:
        continue

    fusion_row = metrics_by_variant['multimodal_fusion']
    summary_row = {
        'exercise': cfg['exercise'],
        'seq_len': cfg['seq_len'],
        'rgb_reference_variant': cfg['rgb_reference_variant'],
        'model_mae_fusion': fusion_row['model_mae'],
        'model_within_1_fusion': fusion_row['model_within_1'],
        'delta_mae_vs_trivial_fusion': fusion_row['delta_mae_vs_trivial'],
        'delta_within_1_vs_trivial_fusion': fusion_row['delta_within_1_vs_trivial'],
    }
    if 'pose_6B' in metrics_by_variant:
        pose_row = metrics_by_variant['pose_6B']
        summary_row['model_mae_pose_6B'] = pose_row['model_mae']
        summary_row['model_within_1_pose_6B'] = pose_row['model_within_1']
        summary_row['delta_mae_fusion_minus_pose_6B'] = fusion_row['model_mae'] - pose_row['model_mae']
        summary_row['delta_within_1_fusion_minus_pose_6B'] = fusion_row['model_within_1'] - pose_row['model_within_1']
    if cfg['rgb_reference_variant'] in metrics_by_variant:
        rgb_ref_row = metrics_by_variant[cfg['rgb_reference_variant']]
        summary_row['model_mae_rgb_reference'] = rgb_ref_row['model_mae']
        summary_row['model_within_1_rgb_reference'] = rgb_ref_row['model_within_1']
        summary_row['delta_mae_fusion_minus_rgb_reference'] = fusion_row['model_mae'] - rgb_ref_row['model_mae']
        summary_row['delta_within_1_fusion_minus_rgb_reference'] = fusion_row['model_within_1'] - rgb_ref_row['model_within_1']
    if 'pose_best_squat' in metrics_by_variant:
        best_pose_row = metrics_by_variant['pose_best_squat']
        summary_row['model_mae_pose_best_squat'] = best_pose_row['model_mae']
        summary_row['model_within_1_pose_best_squat'] = best_pose_row['model_within_1']
        summary_row['delta_mae_fusion_minus_best_squat'] = fusion_row['model_mae'] - best_pose_row['model_mae']
        summary_row['delta_within_1_fusion_minus_best_squat'] = fusion_row['model_within_1'] - best_pose_row['model_within_1']
    summary_rows.append(summary_row)

baseline_df = pd.DataFrame(rows)
display(baseline_df.sort_values(['exercise', 'variant']))
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    display(summary_df.sort_values('exercise'))
if comparison_failures:
    display(pd.DataFrame(comparison_failures))
